# ライブラリのインポート

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore")
import random
import polars as pl
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import torch
from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

# 変数の設定

In [ ]:
class CFG:
    VER = 1
    AUTHOR = "suba"
    COMPETITION = "atmacup17"
    DATA_PATH = Path("/workspace/kaggle_llm_book/ch3/data/takaito/atmacup17")
    MODEL_PATH = "microsoft/deberta-v3-large"  # ヒント: HuggingFace上のDeBERTa v3-largeのモデルID
    MODEL_NAME = "deberta-v3-large"
    MAX_LENGTH = 2048  # ヒント: トークン列の最大長（2のべき乗が一般的）
    EPOCHS = 3  # ヒント: 学習エポック数
    STEPS = 20
    WARMUP_RATIO = 0.05
    TRAIN_BATCH_SPLIT = 8
    TRAIN_BATCH_SIZE = 64
    EVAL_BATCH_SIZE = 64
    LEARNING_RATE = 2e-5  # ヒント: fine-tuningの典型的な学習率（2e-5など）
    OPTIM = "adamw_torch"
    USE_GPU = torch.cuda.is_available()
    SEED = 42
    N_SPLIT = 3  # ヒント: 交差検証のfold数
    TARGET_COL = "Recommended IND"  # ヒント: 目的変数のカラム名（"Recommended IND"）
    TARGET_COL_CLASS_NUM = 2
    METRIC = "auc"  # ヒント: 評価指標名（"auc"）
    METRIC_MAXIMIZE_FLAG = True
    PREFIX  = f"{AUTHOR}_{MODEL_NAME}_seed{SEED}_ver{VER}"
    OUTPUT_DIR = "./model"

# 乱数の設定

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if CFG.USE_GPU:
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)

# デバイスの設定

In [ ]:
device = torch.device("cuda" if CFG.USE_GPU else "cpu")
print(device)

# データの読み込み

In [ ]:
clothing_master_df = pl.read_csv(CFG.DATA_PATH / "clothing_master.csv")
train_df = pl.read_csv(CFG.DATA_PATH / "train.csv")
test_df = pl.read_csv(CFG.DATA_PATH / "test.csv")

# データの加工

In [ ]:
def make_text_column(df):
    # "Title" と "Review Text" を空白で結合して "text" 列を作る
    df = df.with_columns(
        (pl.col("Title") + " " + pl.col("Review Text")).alias("text")
    )
    return df

def preprocessing(df, clothing_master_df):
    # "Title" と "Review Text" の欠損値を空文字列で埋める
    df = df.with_columns([
        pl.col("Title").fill_null(""),
        pl.col("Review Text").fill_null(""),
    ])
    # clothing_master_df を "Clothing ID" キーで左結合する
    df = df.join(clothing_master_df, on="Clothing ID", how="left")
    # make_text_column() を呼んで text 列を作る
    df = make_text_column(df)
    return df

In [ ]:
train_df = preprocessing(train_df, clothing_master_df)
test_df = preprocessing(test_df, clothing_master_df)
# TARGET_COL を int32 型に変換して "labels" 列として追加
train_df = train_df.with_columns(
    pl.col(CFG.TARGET_COL).cast(pl.Int32).alias("labels")
)

# トークナイザの設定

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_PATH)

def tokenize(sample, tokenizer):
    # tokenizer を使って sample["text"] をトークナイズする
    return tokenizer(sample["text"], max_length=CFG.MAX_LENGTH, truncation=True)

# データセットの作成

In [ ]:
def mk_dataset(df, tokenizer):
    # polars DataFrame を dict 経由で HuggingFace Dataset に変換する
    raw_datasets = Dataset.from_dict(df.to_dict(as_series=False))
    tokenized_datasets = raw_datasets.map(
        tokenize,
        fn_kwargs={"tokenizer": tokenizer}
    ).remove_columns(["text"])
    return tokenized_datasets

In [ ]:
ds = mk_dataset(train_df.select(["text", "labels"]), tokenizer)
ds_test = mk_dataset(test_df.select(["text"]), tokenizer)

In [ ]:
print(ds)

In [ ]:
print(ds["input_ids"][0])

# モデルの学習と推論

In [ ]:
def compute_metrics(p):
    label_preds, labels = p
    label_prob_preds = torch.softmax(torch.tensor(label_preds), dim=1).numpy()[:, 1]
    score = roc_auc_score(labels, label_prob_preds)
    return {"auc": score}

In [ ]:
def get_args(CFG, fold):
    args = TrainingArguments(
        output_dir=f"./model/{CFG.PREFIX}-fold{fold}",
        report_to="none",
        eval_strategy="steps",
        logging_strategy="steps",
        save_strategy="steps",
        eval_steps=CFG.STEPS,
        logging_steps=CFG.STEPS,
        save_steps=CFG.STEPS,
        save_total_limit=1,
        metric_for_best_model=CFG.METRIC,
        greater_is_better=CFG.METRIC_MAXIMIZE_FLAG,
        optim=CFG.OPTIM,
        learning_rate=CFG.LEARNING_RATE,
        lr_scheduler_type="linear",
        warmup_ratio=CFG.WARMUP_RATIO,
        per_device_train_batch_size=CFG.TRAIN_BATCH_SIZE // CFG.TRAIN_BATCH_SPLIT,
        per_device_eval_batch_size=CFG.EVAL_BATCH_SIZE,
        gradient_accumulation_steps=CFG.TRAIN_BATCH_SPLIT,
        num_train_epochs=CFG.EPOCHS,
        fp16=CFG.USE_GPU,
        seed=CFG.SEED,
        data_seed=CFG.SEED,
    )
    return args

In [ ]:
predictions = np.zeros(len(train_df))
test_predictions = np.zeros(len(test_df))

# 交差検証のために、目的変数が各 fold で均一になるようにデータを分割
kfold = StratifiedKFold(n_splits=CFG.N_SPLIT, shuffle=True, random_state=CFG.SEED)

for fold, (train_index, valid_index) in enumerate(
    kfold.split(train_df.to_numpy(), train_df[CFG.TARGET_COL].to_numpy())
):
    # モデルの読み込み & 設定
    config = AutoConfig.from_pretrained(CFG.MODEL_PATH)
    config.num_labels = CFG.TARGET_COL_CLASS_NUM
    model = AutoModelForSequenceClassification.from_pretrained(CFG.MODEL_PATH, config=config)

    trainer = Trainer(
        model=model,
        args=get_args(CFG, fold),
        train_dataset=ds.select(train_index),
        eval_dataset=ds.select(valid_index),
        data_collator=DataCollatorWithPadding(tokenizer),
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    # 学習開始
    trainer.train()

    # モデルなどの保存
    trainer.save_model(f"{CFG.PREFIX}/{CFG.PREFIX}-fold{fold}")
    tokenizer.save_pretrained(f"{CFG.PREFIX}/{CFG.PREFIX}-fold{fold}")

    # 検証セットの推論
    valid_preds = trainer.predict(ds.select(valid_index)).predictions
    predictions[valid_index] = torch.softmax(torch.tensor(valid_preds), dim=1).numpy()[:, 1]

    # テストセットの推論
    test_preds = trainer.predict(ds_test).predictions
    test_predictions += torch.softmax(torch.tensor(test_preds), dim=1).numpy()[:, 1] / CFG.N_SPLIT

In [ ]:
print("CV Score: ", roc_auc_score(train_df[CFG.TARGET_COL].to_list(), predictions))

In [ ]:
train_df = train_df.with_columns(pl.Series(f"{CFG.PREFIX}_pred_prob", predictions))
train_df.select([f"{CFG.PREFIX}_pred_prob"]).write_csv(f"{CFG.PREFIX}_train_preds.csv")
test_df = test_df.with_columns(pl.Series(f"{CFG.PREFIX}_pred_prob", test_predictions))
test_df.select([f"{CFG.PREFIX}_pred_prob"]).write_csv(f"{CFG.PREFIX}_test_preds.csv")

In [ ]:
test_df = test_df.with_columns(pl.col(f"{CFG.PREFIX}_pred_prob").alias("target"))
test_df.select(["target"]).write_csv(f"{CFG.PREFIX}_submission.csv")